# Making a Medical Imaging App

The following jupyter notebook goes through my learning path of making my own medical imaging app. Although existing apps like horos and weasel exist (and they work well) I want to make my own to learn essential coding skills, but also make it alot more open source, so me and furture developers can integrate further funcitonalities and AI integration. But mainly, its just for me to learn.

### Simple DICOM Viewer
**What this does:**
- Takes a path do a DICOM path
- Extracts patient metadata/pixel array
- Calculates physical size of scan
- Uses matplotlib to plot the pixel array into an image
- Renders the image clearly with a scale 

**What I want in a next iteration:**
- Change contrast
- Zooom in, pan (Better Viewer and UI)
- Have the ability to select a region of interest, have that sent to an LLM with the main picture as context

In [ ]:
import pydicom
import matplotlib.pyplot as plt

"""
Here im setting a variable path as a string to my path to the DICOM file. 
In this case its a sinlgle abdominal XRAY to check for post surgery constipation.

ds = pydicom.dcmread(path) sets the variable ds to read the DIOCM file.
In short: .dcmread reads through the binary in the DICOM file and first extracts all the metadata (like patient info etc)
Then saves that as a FileDataset Class (A fancy dictionary). When .dcmread gets to the Pixel Data tag:
It doesnt decode the image matrix immediatley, but instead stores it in .PixelData. So to get the matrix you just get ds.PixelData
"""
path = "/Users/hamza/Code/Medimaging/IN000001"
ds = pydicom.dcmread(path)

"""
Sets a constant PIXEL_ARRAY as the numpy image matrix
Extracts No. rows/columns from ds
"""
# Extract Global Constants (Immutable Data)
PIXEL_ARRAY = ds.pixel_array
N_ROWS = ds.Rows
N_COLS = ds.Columns

"""
Extracts row/columns spacing. The spacing is the physical mm distance between the centre of one image pixel to another.
This can be used to comnvert No. Pixels to a physical mm size
"""
# Spacing: [Row (Y), Column (X)]
ROW_SPACING = ds.PixelSpacing[0] 
COL_SPACING = ds.PixelSpacing[1] 

"""
A seperate function to calulate physical size from number of rows/colums and spacing
"""
def calculate_physical_size():
    calc_height = N_ROWS * ROW_SPACING
    calc_width  = N_COLS * COL_SPACING
    return calc_height, calc_width

"""
This is key, we set the two variables here so they are global.
"""
# Capture the physical dimensions globally
PHYSICAL_HEIGHT, PHYSICAL_WIDTH = calculate_physical_size()

"""
def inspect_metadata exists so i can simply change verbose True/False to show the key metadata i want when i want
"""
def inspect_metadata(verbose=True):
    if not verbose:
        return
        
    print(f"{' Metadata Inspection ':—^40}")
    print(f"Matrix Size (Px)   : {N_ROWS} x {N_COLS}")
    print(f"Pixel Spacing (mm) : {ROW_SPACING:.3f} x {COL_SPACING:.3f}")
    print(f"Field of View (mm) : {PHYSICAL_WIDTH:.1f} x {PHYSICAL_HEIGHT:.1f}")
    print("—" * 40)

"""
This Funciton does the actual rendering. It converts ds.PixelData to an image with axis and everything.
It takes 
"""
def render_radiology_view(data, physical_dims=None, cmap='gray'):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=100)
    extent_limit = None
    xlabel, ylabel = "Pixels", "Pixels"
    
    if physical_dims:
        h_mm, w_mm = physical_dims
        extent_limit = [0, w_mm, h_mm, 0] 
        xlabel, ylabel = "Width (mm)", "Height (mm)"

    im = ax.imshow(data, cmap=cmap, extent=extent_limit, origin='upper')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Signal Intensity (Raw)')

    ax.set_title(f"DICOM Viewer | {data.shape[1]}x{data.shape[0]} Matrix", fontsize=12, pad=10)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)

    plt.tight_layout()
    plt.show()

inspect_metadata(verbose=True)

# We pack the dims into a tuple to pass them cleanly
dims_tuple = (PHYSICAL_HEIGHT, PHYSICAL_WIDTH)

# Launch the view
render_radiology_view(PIXEL_ARRAY, physical_dims=dims_tuple)

### Better DICOM Viewer (Uses SimpleITK )
**What this does:**
- Same functions as previous

**What I want in a next iteration:**
- Change contrast
- Zooom in, pan (Better Viewer and UI)
- Have the ability to select a region of interest, have that sent to an LLM with the main picture as context

In [ ]:
import SimpleITK as sitk
import matplotlib.pyplot as plt

# 1. SETUP
# Make sure this path points to a valid file on your machine
path = "/Users/hamza/Code/Medimaging/IN000001"
image = sitk.ReadImage(path)

# 2. EXTRACT DATA
pixel_array = sitk.GetArrayFromImage(image).squeeze()

# 3. GET PHYSICAL DIMENSIONS
spacing = image.GetSpacing()  # Returns tuple: (x_spacing, y_spacing)
size = image.GetSize()        # Returns tuple: (width, height)

def calculate_physical_size():
    # size[0] is width (x), size[1] is height (y)
    calc_width_mm = size[0] * spacing[0]
    calc_height_mm = size[1] * spacing[1]
    return calc_width_mm, calc_height_mm

# FIX: Unpack in the correct order (Width, Height) to match the return statement
phys_width, phys_height = calculate_physical_size()

def inspect_metadata(verbose=True):
    if not verbose:
        return
    
    # FIX: Replaced em-dash '—' with standard hyphen '-'
    print(f"{' Metadata Inspection ':-^40}")
    
    # FIX: Cannot format a tuple directly. Accessed elements [0] and [1].
    print(f"Pixel Spacing (mm) : {spacing[0]:.3f} x {spacing[1]:.3f}")
    
    print(f"Field of View (mm) : {phys_width:.1f} x {phys_height:.1f}")
    print("-" * 40)

def render_radiology_view(data, physical_dims=None, cmap='gray'):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=100)
    extent_limit = None
    xlabel, ylabel = "Pixels", "Pixels"
    
    if physical_dims:
        # Expecting tuple (height_mm, width_mm)
        h_mm, w_mm = physical_dims
        
        # Extent format: [left, right, bottom, top]
        # Since origin is 'upper', 'bottom' is the max Y value (height)
        extent_limit = [0, w_mm, h_mm, 0] 
        xlabel, ylabel = "Width (mm)", "Height (mm)"

    im = ax.imshow(data, cmap=cmap, extent=extent_limit, origin='upper')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Signal Intensity (Raw)')

    # data.shape is (Rows/Height, Cols/Width) in Numpy
    ax.set_title(f"DICOM Viewer | {data.shape[1]}x{data.shape[0]} Matrix", fontsize=12, pad=10)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)

    plt.tight_layout()
    plt.show()

# ==========================================
# 5. EXECUTION FLOW
# ==========================================
inspect_metadata(verbose=True)

# FIX: Use correct variable names (lower case) defined earlier
# We pass (Height, Width) to match the unpacking inside render_radiology_view
dims_tuple = (phys_height, phys_width)

# FIX: Use 'pixel_array' (defined at top), not 'PIXEL_ARRAY'
render_radiology_view(pixel_array, physical_dims=dims_tuple)

### Better DICOM Viewer (Uses SimpleITK ) AND Has a Better UI
**What this does:**
- Same functions as previous

**What I want in a next iteration:**
- Change contrast
- Zooom in, pan (Better Viewer and UI)
- Have the ability to select a region of interest, have that sent to an LLM with the main picture as context

In [ ]:
import SimpleITK as sitk
import matplotlib.pyplot as plt

# 1. SETUP
# Make sure this path points to a valid file on your machine
path = "/Users/hamza/Code/Medimaging/IN000001"
image = sitk.ReadImage(path)

# 2. EXTRACT DATA
pixel_array = sitk.GetArrayFromImage(image).squeeze()

# 3. GET PHYSICAL DIMENSIONS
spacing = image.GetSpacing()  # Returns tuple: (x_spacing, y_spacing)
size = image.GetSize()        # Returns tuple: (width, height)

def calculate_physical_size():
    # size[0] is width (x), size[1] is height (y)
    calc_width_mm = size[0] * spacing[0]
    calc_height_mm = size[1] * spacing[1]
    return calc_width_mm, calc_height_mm

# FIX: Unpack in the correct order (Width, Height) to match the return statement
phys_width, phys_height = calculate_physical_size()

def inspect_metadata(verbose=True):
    if not verbose:
        return
    
    # FIX: Replaced em-dash '—' with standard hyphen '-'
    print(f"{' Metadata Inspection ':-^40}")
    
    # FIX: Cannot format a tuple directly. Accessed elements [0] and [1].
    print(f"Pixel Spacing (mm) : {spacing[0]:.3f} x {spacing[1]:.3f}")
    
    print(f"Field of View (mm) : {phys_width:.1f} x {phys_height:.1f}")
    print("-" * 40)

def render_radiology_view(data, physical_dims=None, cmap='gray'):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=100)
    extent_limit = None
    xlabel, ylabel = "Pixels", "Pixels"
    
    if physical_dims:
        # Expecting tuple (height_mm, width_mm)
        h_mm, w_mm = physical_dims
        
        # Extent format: [left, right, bottom, top]
        # Since origin is 'upper', 'bottom' is the max Y value (height)
        extent_limit = [0, w_mm, h_mm, 0] 
        xlabel, ylabel = "Width (mm)", "Height (mm)"

    im = ax.imshow(data, cmap=cmap, extent=extent_limit, origin='upper')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Signal Intensity (Raw)')

    # data.shape is (Rows/Height, Cols/Width) in Numpy
    ax.set_title(f"DICOM Viewer | {data.shape[1]}x{data.shape[0]} Matrix", fontsize=12, pad=10)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)

    plt.tight_layout()
    plt.show()

# ==========================================
# 5. EXECUTION FLOW
# ==========================================
inspect_metadata(verbose=True)

# FIX: Use correct variable names (lower case) defined earlier
# We pass (Height, Width) to match the unpacking inside render_radiology_view
dims_tuple = (phys_height, phys_width)

# FIX: Use 'pixel_array' (defined at top), not 'PIXEL_ARRAY'
render_radiology_view(pixel_array, physical_dims=dims_tuple)